In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType, TimestampType,
)
import os, shutil

# Spark ML — transformers used inside foreachBatch
from pyspark.ml.feature import VectorAssembler, Normalizer, Tokenizer, HashingTF
from pyspark.ml.stat import Summarizer

spark = SparkSession.builder \
    .appName("RetailStreaming") \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.sql.streaming.schemaInference", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 21:30:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session ready.


## Structured Streaming — Simulated Retail Transaction Stream

Spark Structured Streaming treats a live data source as an unbounded table that grows over time. Each micro-batch reads new rows, runs the query incrementally, and updates the result table.

Our dataset is static (a Parquet file), so we simulate a stream by splitting it into 30 small Parquet files and placing them in a watched directory. Spark's file-source reader picks them up one by one, exactly as it would if they were arriving from a live pipeline.

We demonstrate three core streaming concepts:
- **Windowed aggregation** — group events into fixed time windows (monthly revenue)
- **Watermarking** — tell Spark how late data can arrive before a window is finalised
- **`trigger(availableNow=True)`** — process all currently available data then stop, which is the clean way to run a bounded stream in a notebook

In [2]:
# Load and clean the retail data the same way as ml.ipynb
raw_df = spark.read.parquet("../Data/online_retail_II.parquet")

master_df = (
    raw_df
    .filter(~F.col("Invoice").startswith("C"))
    .filter(F.col("Quantity") > 0)
    .filter(F.col("Price") > 0)
    .dropna(subset=["Customer ID"])
    .fillna({"Description": "unknown"})
    .withColumn("Quantity", F.col("Quantity").cast(DoubleType()))
    .withColumn("Revenue", F.col("Quantity") * F.col("Price"))
)

# Split into 30 partitions and write to a temp directory.
# Each partition file becomes one "arrival" in the simulated stream.
STREAM_DIR = "/tmp/retail_stream_source"
if os.path.exists(STREAM_DIR):
    shutil.rmtree(STREAM_DIR)

master_df.repartition(30).write.parquet(STREAM_DIR)

file_count = len([f for f in os.listdir(STREAM_DIR) if f.endswith(".parquet")])
print(f"Source directory: {STREAM_DIR}")
print(f"Mini-batch files written: {file_count}")
print(f"Total records: {master_df.count():,}")

Source directory: /tmp/retail_stream_source
Mini-batch files written: 30
Total records: 805,549


In [3]:
# Streaming readers require an explicit schema — inferSchema is not supported
# because Spark cannot afford to scan the full source just to derive types.
stream_schema = StructType([
    StructField("Invoice",     StringType(),       True),
    StructField("StockCode",   StringType(),       True),
    StructField("Description", StringType(),       True),
    StructField("Quantity",    DoubleType(),        True),
    StructField("InvoiceDate", TimestampType(),    True),
    StructField("Price",       DoubleType(),        True),
    StructField("Customer ID", DoubleType(),        True),
    StructField("Country",     StringType(),       True),
    StructField("Revenue",     DoubleType(),        True),
])

stream_df = (
    spark.readStream
    .format("parquet")
    .schema(stream_schema)
    .load(STREAM_DIR)
)

print(f"Streaming DataFrame created  |  isStreaming = {stream_df.isStreaming}")
stream_df.printSchema()

Streaming DataFrame created  |  isStreaming = True
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Revenue: double (nullable = true)



## Query 1 — Windowed Monthly Revenue

We group events into 30-day tumbling windows and sum revenue within each window. The watermark of 7 days tells Spark to wait up to 7 days for late-arriving events before considering a window closed. Once a window is finalised, it will not be updated even if late data arrives.

We use `outputMode("complete")` so every micro-batch outputs the full current state of all windows — convenient for inspecting results in a notebook. The results land in a Spark in-memory table (`queryName`) that we can query with SQL.

In [4]:
monthly_agg = (
    stream_df
    .withWatermark("InvoiceDate", "7 days")
    .groupBy(F.window("InvoiceDate", "30 days").alias("window_30d"))
    .agg(
        F.round(F.sum("Revenue"), 2).alias("total_revenue"),
        F.approx_count_distinct("Invoice").alias("num_invoices_approx"),
    )
)

q1 = (
    monthly_agg.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("monthly_revenue_stream")
    .trigger(availableNow=True)
    .start()
)

q1.awaitTermination()
print("Query 1 finished.")
print(f"Status: {q1.status['message']}")

26/05/17 21:30:32 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d08958a3-271c-4098-94d5-7ab508bdc7b7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/17 21:30:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/17 21:30:32 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/05/17 21:30:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Query 1 finished.
Status: Stopped


In [5]:
spark.sql("""
    SELECT
        date_format(window_30d.start, 'yyyy-MM-dd') AS window_start,
        date_format(window_30d.end,   'yyyy-MM-dd') AS window_end,
        total_revenue,
        num_invoices_approx
    FROM monthly_revenue_stream
    ORDER BY window_start
""").show(30, truncate=False)

+------------+----------+-------------+-------------------+
|window_start|window_end|total_revenue|num_invoices_approx|
+------------+----------+-------------+-------------------+
|2009-11-02  |2009-12-02|44048.69     |97                 |
|2009-12-02  |2010-01-01|642605.47    |1410               |
|2010-01-01  |2010-01-31|538952.87    |944                |
|2010-01-31  |2010-03-02|548719.93    |1257               |
|2010-03-02  |2010-04-01|675626.32    |1475               |
|2010-04-01  |2010-05-01|594609.19    |1346               |
|2010-05-01  |2010-05-31|599985.79    |1441               |
|2010-05-31  |2010-06-30|607486.67    |1347               |
|2010-06-30  |2010-07-30|608825.31    |1377               |
|2010-07-30  |2010-08-29|581260.73    |1191               |
|2010-08-29  |2010-09-28|754145.87    |1577               |
|2010-09-28  |2010-10-28|1054246.5    |2204               |
|2010-10-28  |2010-11-27|1128642.82   |2651               |
|2010-11-27  |2010-12-27|1025561.0    |1

## Query 2 — Running Country Revenue Totals

A simpler aggregation without windowing — running totals of revenue and transaction count per country across the full stream. This shows how a live dashboard could track which markets are growing in real time as new transactions arrive.

In [6]:
country_agg = (
    stream_df
    .groupBy("Country")
    .agg(
        F.round(F.sum("Revenue"), 2).alias("total_revenue"),
        F.count("*").alias("num_transactions"),
    )
)

q2 = (
    country_agg.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("country_revenue_stream")
    .trigger(availableNow=True)
    .start()
)

q2.awaitTermination()
print("Query 2 finished.")

26/05/17 21:30:37 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-73ac0826-fd2a-4085-b03c-af0a74127ebb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/17 21:30:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/17 21:30:37 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


Query 2 finished.


In [7]:
spark.sql("""
    SELECT Country, total_revenue, num_transactions
    FROM country_revenue_stream
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

+--------------+-------------+----------------+
|Country       |total_revenue|num_transactions|
+--------------+-------------+----------------+
|United Kingdom|1.472314752E7|725250          |
|EIRE          |621631.11    |15743           |
|Netherlands   |554232.34    |5088            |
|Germany       |431262.46    |16694           |
|France        |355257.47    |13812           |
|Australia     |169968.11    |1812            |
|Spain         |109178.53    |3719            |
|Switzerland   |100365.34    |3011            |
|Sweden        |91549.72     |1319            |
|Denmark       |69862.19     |798             |
+--------------+-------------+----------------+



## Query 3 — foreachBatch: Custom Processing Per Micro-Batch

`foreachBatch` gives full control over what happens with each micro-batch — the batch DataFrame can be written to multiple sinks, passed to a Spark ML model for scoring, or processed with arbitrary logic. Here we use it to identify the top-spending customer in each micro-batch and accumulate the results.

In [8]:
# Accumulate the top customer from each batch into a plain Python list.
# This is driver-side state — fine for a small summary, not for large state.
top_customers_log = []

def process_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return
    top = (
        batch_df
        .groupBy(F.col("`Customer ID`"))
        .agg(F.round(F.sum("Revenue"), 2).alias("batch_revenue"))
        .orderBy(F.desc("batch_revenue"))
        .first()
    )
    if top:
        top_customers_log.append({
            "batch_id":    batch_id,
            "customer_id": top["Customer ID"],
            "revenue":     top["batch_revenue"],
        })

q3 = (
    stream_df.writeStream
    .outputMode("append")
    .foreachBatch(process_batch)
    .trigger(availableNow=True)
    .start()
)

q3.awaitTermination()
print(f"Query 3 finished — processed {len(top_customers_log)} batches.")

26/05/17 21:30:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bd4b287a-55a5-49df-af33-552fe17ca243. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/17 21:30:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query 3 finished — processed 1 batches.


In [9]:
# Show the top-spending customer from each of the first 10 micro-batches
print(f"{'Batch':>6}  {'Customer ID':>12}  {'Batch Revenue (GBP)':>20}")
print("-" * 44)
for row in top_customers_log[:10]:
    print(f"  {row['batch_id']:>4}  {row['customer_id']:>12.0f}  {row['revenue']:>20,.2f}")

 Batch   Customer ID   Batch Revenue (GBP)
--------------------------------------------
     0         18102            608,821.65


## Query 4 — ML Feature Extraction inside foreachBatch

Because `foreachBatch` exposes each micro-batch as a plain Spark DataFrame, we can run any Spark ML transformer on it directly. Here we apply two pipelines from the course on every arriving batch:

- **`VectorAssembler` → `Normalizer` → `Summarizer`** (Weeks 7–8): assemble Quantity, Price, and Revenue into a feature vector, L2-normalise it, then compute per-batch mean statistics.
- **`Tokenizer` → `HashingTF`** (Weeks 7–8): tokenise product descriptions and hash them to a TF vector, then identify the highest-revenue product description in each batch.

Transformers (unlike estimators) need no fitting step, so they are safe to call inside a streaming batch handler without any data leakage risk.

In [10]:
# Transformers are stateless — instantiate once, reuse across all batches
vec_assembler = VectorAssembler(
    inputCols=["Quantity", "Price", "Revenue"],
    outputCol="num_features",
    handleInvalid="skip",
)
normalizer  = Normalizer(inputCol="num_features", outputCol="norm_features", p=2.0)
tokenizer   = Tokenizer(inputCol="Description", outputCol="tokens")
hashing_tf  = HashingTF(inputCol="tokens", outputCol="tf_features", numFeatures=128)
stat_metric = Summarizer.metrics("mean", "count")

batch_summaries = []   # collect one row per batch on the driver

def ml_batch_handler(batch_df, batch_id):
    if batch_df.count() == 0:
        return

    # Numeric path: assemble -> normalise -> summarise
    vec_df  = vec_assembler.transform(batch_df)
    norm_df = normalizer.transform(vec_df)
    stats   = norm_df.select(
        stat_metric.summary(F.col("norm_features"))
    ).first()[0]

    # Text path: tokenise descriptions -> hash to TF vector
    tok_df = tokenizer.transform(batch_df)
    tf_df  = hashing_tf.transform(tok_df)

    # Top product by revenue in this batch (safe: result is 1 row)
    top = (
        tf_df
        .groupBy("Description")
        .agg(F.round(F.sum("Revenue"), 2).alias("batch_revenue"))
        .orderBy(F.desc("batch_revenue"))
        .first()
    )

    batch_summaries.append({
        "batch":       batch_id,
        "n_rows":      int(stats["count"]),
        "mean_qty":    round(float(stats.mean[0]), 4),
        "mean_price":  round(float(stats.mean[1]), 4),
        "mean_rev":    round(float(stats.mean[2]), 4),
        "top_product": top["Description"] if top else "",
        "top_revenue": top["batch_revenue"] if top else 0.0,
    })

q4 = (
    stream_df.writeStream
    .outputMode("append")
    .foreachBatch(ml_batch_handler)
    .trigger(availableNow=True)
    .start()
)

q4.awaitTermination()
print(f"Query 4 finished — {len(batch_summaries)} batches processed.")

26/05/17 21:30:42 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6c135ff7-d0d2-43cb-8060-75bf20f61c33. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/17 21:30:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query 4 finished — 1 batches processed.


In [11]:
# Show per-batch ML summary: normalised feature means and top product
print(f"{'Batch':>5}  {'Rows':>6}  {'Mean Qty':>9}  {'Mean Price':>10}  "
      f"{'Mean Rev':>9}  Top Product")
print("-" * 90)
for s in batch_summaries[:10]:
    print(f"  {s['batch']:>3}  {s['n_rows']:>6}  {s['mean_qty']:>9.4f}  "
          f"{s['mean_price']:>10.4f}  {s['mean_rev']:>9.4f}  "
          f"{s['top_product'][:35]}")

Batch    Rows   Mean Qty  Mean Price   Mean Rev  Top Product
------------------------------------------------------------------------------------------
    0  805549     0.4581      0.2575     0.7573  REGENCY CAKESTAND 3 TIER


In [12]:
# Stop all active streaming queries cleanly
active = spark.streams.active
for q in active:
    q.stop()
    print(f"Stopped query: {q.name}")

# Remove the temporary source directory
shutil.rmtree(STREAM_DIR, ignore_errors=True)
print(f"Cleaned up {STREAM_DIR}")

Cleaned up /tmp/retail_stream_source


## Why Structured Streaming is Valuable Here

The retailer currently processes transactions in batch — reports are generated once a day from a full reload. With Structured Streaming, the same aggregations (monthly revenue, country totals) could run continuously, giving the business near-real-time visibility into sales performance.

The `foreachBatch` pattern in Query 3 shows how a pre-trained Spark ML model (such as the GBT classifier from the ml notebook) could score each arriving transaction immediately, enabling live anomaly detection or dynamic pricing without waiting for a nightly batch run.

Operationally, the file-source pattern we used here maps directly to S3 or ADLS landing zones, where upstream systems drop files and Spark picks them up automatically — no custom consumer code required.